# install dependencies

In [21]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
import statsmodels.api as sm
import pyreadstat

# Phase1 Data Preparation

In [22]:
def calculate_cir_series(
    outcome,
    file_name,
    data_path='../../result/occ/analysis',
    horizon_start=1,
    horizon_end=36,
    series_name=None
):
    file_path = f"{data_path.rstrip('/\\')}/{file_name}"
    df = pd.read_csv(file_path)

    if series_name is None:
        series_name = f"CIR_{horizon_end}"

    df_outcome = df[df['outcome'] == outcome].copy()
    df_outcome['horizon'] = pd.to_numeric(df_outcome['horizon'])
    df_outcome = df_outcome.sort_values('horizon').reset_index(drop=True)

    group_cols = [col for col in df_outcome.columns if col.startswith('group')]

    cir_dict = {}
    for col in group_cols:
        irf_window = df_outcome[df_outcome['horizon'].between(horizon_start, horizon_end)][col]
        cir_dict[col] = irf_window.sum()

    cir_series = pd.Series(cir_dict, name=series_name)
    return cir_series

In [23]:
def build_y_series_from_mapping(
    cir_series,
    file_name,
    data_path='../../result/mapping',
    sheet_name='Sheet1',
    usecols='A,E',
    series_name=None
):
    mapping_path = f"{data_path.rstrip('/\\')}/{file_name}"
    df_map = pd.read_excel(mapping_path, sheet_name=sheet_name, usecols=usecols, header=0)
    df_map.columns = ['occ1990', 'SOC-2018']

    if series_name is None:
        series_name = cir_series.name if cir_series.name is not None else 'value'

    df_map['occ1990'] = pd.to_numeric(df_map['occ1990'], errors='coerce').astype('Int64')
    df_map['SOC-2018'] = df_map['SOC-2018'].astype(str).str.strip()
    df_map = df_map.dropna(subset=['occ1990', 'SOC-2018'])

    def occ1990_to_group(occ):
        if 3 <= occ <= 37: return 1
        elif 43 <= occ <= 200: return 2
        elif 203 <= occ <= 235: return 3
        elif 243 <= occ <= 283: return 4
        elif 303 <= occ <= 389: return 5
        elif 405 <= occ <= 469: return 6
        elif (473 <= occ <= 498) or (558 <= occ <= 599) or (614 <= occ <= 617): return 7
        elif (503 <= occ <= 549) or (628 <= occ <= 699): return 8
        elif (703 <= occ <= 799) or (803 <= occ <= 889): return 9
        return np.nan

    df_map['group'] = df_map['occ1990'].apply(occ1990_to_group)

    group_to_value = {}
    for idx, val in cir_series.items():
        match = re.search(r'group(\d+)', str(idx))
        if match:
            group_to_value[int(match.group(1))] = val

    df_map[series_name] = df_map['group'].map(group_to_value)
    df_final = df_map.drop_duplicates(subset='SOC-2018', keep='first')
    y_series = df_final.set_index('SOC-2018')[series_name].dropna()

    return y_series

In [24]:
def load_and_prepare_onet_data_extended(
    y_series,
    file_names,
    rti_path,
    mapping_path,
    onet_data_path='../../data/ONET',
    mapping_sheet='Sheet1',
    scale_id='LV',
    usecols=[0, 1, 4, 5, 7]
):
    import pyreadstat

    # ── 1. 读取 RTI，以 occ1990dd 为基准 ─────────────────────
    rti_raw, _ = pyreadstat.read_dta(rti_path)
    rti_raw = pd.DataFrame(rti_raw)[['occ1990dd', 'task_abstract', 'task_routine', 'task_manual']]
    rti_raw['occ1990dd'] = rti_raw['occ1990dd'].astype(int)
    rti_raw['RTI'] = (
        np.log(rti_raw['task_routine'].clip(lower=1e-6)) -
        np.log(rti_raw['task_manual'].clip(lower=1e-6))  -
        np.log(rti_raw['task_abstract'].clip(lower=1e-6))
    )
    valid_occ1990dd = set(rti_raw['occ1990dd'].unique())
    print(f"RTI 文件中有效 occ1990dd: {len(valid_occ1990dd)} 个")

    # ── 2. 读取 mapping，只保留 occ1990dd 在 RTI 中的行 ──────
    df_map = pd.read_excel(mapping_path, sheet_name=mapping_sheet, usecols='A,B,E', header=0)
    df_map.columns = ['occ1990', 'occ1990dd', 'SOC_2018']
    df_map = df_map.dropna(subset=['occ1990', 'occ1990dd', 'SOC_2018'])
    df_map['occ1990dd'] = pd.to_numeric(df_map['occ1990dd'], errors='coerce').astype('Int64')
    df_map['SOC_2018']  = df_map['SOC_2018'].astype(str).str.strip()

    # 只保留 occ1990dd 有 RTI 的行
    df_map = df_map[df_map['occ1990dd'].isin(valid_occ1990dd)].copy()
    print(f"mapping 过滤后剩余行数: {len(df_map)}")

    # occ1990dd → RTI，合并进 mapping
    df_map = df_map.merge(rti_raw[['occ1990dd', 'RTI']], on='occ1990dd', how='inner')

    # 同一 SOC_2018 可能对应多个 occ1990dd，取均值
    soc_rti = df_map.groupby('SOC_2018')['RTI'].mean().rename('RTI_index')
    valid_soc = set(soc_rti.index)
    print(f"有 RTI 的 SOC-2018 数量: {len(valid_soc)}")

    # ── 3. 读取四个 O*NET 文件，只保留在 valid_soc 里的 SOC ──
    dfs = []
    for prefix, fname in file_names.items():
        fpath = f"{onet_data_path.rstrip('/\\')}/{fname}"
        df = pd.read_excel(fpath, usecols=usecols, header=0)
        df.columns = ['SOC_Code', 'Sub_Code', 'Element_Name', 'Scale_ID', 'Data_Value']

        df['SOC_Code']     = df['SOC_Code'].astype(str).str.strip()
        df['Element_Name'] = df['Element_Name'].astype(str).str.strip()
        df['Scale_ID']     = df['Scale_ID'].astype(str).str.strip().str.upper()
        df['Sub_Code']     = df['Sub_Code'].astype(str).str.strip().str.zfill(2)

        means = df.groupby(['SOC_Code', 'Element_Name'])['Data_Value'].mean().reset_index()
        means.rename(columns={'Data_Value': 'Mean_Val'}, inplace=True)
        df = df.merge(means, on=['SOC_Code', 'Element_Name'], how='left')
        df.loc[df['Sub_Code'] == '00', 'Data_Value'] = df.loc[df['Sub_Code'] == '00', 'Mean_Val']

        df = df[df['Sub_Code'] == '00'].copy()
        df = df[df['Scale_ID'] == scale_id].copy()
        df = df.dropna(subset=['Data_Value'])
        df.drop(columns=['Mean_Val', 'Sub_Code', 'Scale_ID'], inplace=True)

        # 只保留有 RTI 的 SOC
        df = df[df['SOC_Code'].isin(valid_soc)].copy()
        df['Element_Name'] = prefix + '_' + df['Element_Name']
        dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    df_wide = df_all.pivot_table(
        index='SOC_Code',
        columns='Element_Name',
        values='Data_Value',
        aggfunc='mean'
    ).astype(float)
    df_wide = df_wide.fillna(df_wide.median())
    print(f"O*NET 合并后: {df_wide.shape[0]} 个 SOC, {df_wide.shape[1]} 个特征")

    # ── 4. 拼入 RTI ──────────────────────────────────────────
    df_wide = df_wide.join(soc_rti.rename_axis('SOC_Code'), how='inner')
    print(f"拼入 RTI 后: {df_wide.shape[0]} 个 SOC, {df_wide.shape[1]} 个特征（含 RTI）")

    # ── 5. 标准化 ─────────────────────────────────────────────
    scaler = StandardScaler()
    X_df = pd.DataFrame(
        scaler.fit_transform(df_wide),
        columns=df_wide.columns,
        index=df_wide.index
    )

    # ── 6. 和 y_series 对齐 ───────────────────────────────────
    aligned_idx = X_df.index.intersection(y_series.index)
    X = X_df.loc[aligned_idx].values
    y_aligned = y_series.loc[aligned_idx].values

    print(f"X shape: {X.shape} | Aligned samples: {len(aligned_idx)}")

    return X_df, aligned_idx, X, y_aligned

# Phase2: LASSO Estimation

In [ ]:
def run_lasso_selection(
    X,
    y_aligned,
    feature_names,
    top_n=10,
    n_splits=10,
    random_state=42,
    max_iter=5000
):
    feature_names = pd.Index(feature_names)
    cv_strategy = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    lasso_cv = LassoCV(
        alphas=None,
        cv=cv_strategy,
        max_iter=max_iter,
        random_state=random_state,
        n_jobs=-1
    )
    lasso_cv.fit(X, y_aligned)

    best_alpha = lasso_cv.alpha_
    best_coefs = lasso_cv.coef_
    cv_mse_path = lasso_cv.mse_path_
    cv_mean_mse = cv_mse_path.mean(axis=1)

    nonzero_mask = best_coefs != 0
    nonzero_idx = np.where(nonzero_mask)[0]
    n_nonzero = len(nonzero_idx)

    nonzero_idx_sorted = nonzero_idx[np.argsort(np.abs(best_coefs[nonzero_idx]))[::-1]]
    top_idx = nonzero_idx_sorted[:top_n]

    top_names = feature_names.take(top_idx).tolist()
    top_coefs = best_coefs[top_idx]

    selected_mask = np.zeros(len(feature_names), dtype=bool)
    selected_mask[top_idx] = True

    if n_nonzero < top_n:
        print(f"LASSO only selects {n_nonzero} non-zero variables, fewer than top_n={top_n}, actually using {n_nonzero} variables")

    return {
        'lasso_cv': lasso_cv,
        'best_alpha': best_alpha,
        'best_coefs': best_coefs,
        'cv_mse_path': cv_mse_path,
        'cv_mean_mse': cv_mean_mse,
        'feature_names': feature_names,
        'top_idx': top_idx,
        'top_names': top_names,
        'top_coefs': top_coefs,
        'selected_mask': selected_mask
    }

In [26]:
def lasso_stability_check(
    X,
    y_aligned,
    feature_names,
    top_n=10,
    n_boots=100,
    n_splits=10,
    freq_threshold=0.9,
    random_state=42,
    max_iter=5000
):
    feature_names = pd.Index(feature_names)
    rng = np.random.default_rng(random_state)
    n = len(y_aligned)
    selection_counts = np.zeros(len(feature_names))

    for i in range(n_boots):
        idx = rng.integers(0, n, size=n)
        X_b, y_b = X[idx], y_aligned[idx]
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=int(rng.integers(9999)))
        m = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1).fit(X_b, y_b)
        selection_counts += (m.coef_ != 0).astype(int)

    freq = pd.Series(selection_counts / n_boots, index=feature_names)
    freq = freq.sort_values(ascending=False)

    # 稳定变量：频率 >= freq_threshold
    stable_features = freq[freq >= freq_threshold].index.tolist()
    # 在稳定变量里再截断到 top_n
    final_features = stable_features[:top_n]

    print(f"=== Bootstrap 稳定性检验 (n_boots={n_boots}, threshold={freq_threshold}) ===")
    print(f"频率 >= {freq_threshold} 的变量: {len(stable_features)} 个")
    print(f"频率 >= 0.8 的变量 (高稳定): {(freq >= 0.8).sum()} 个")
    print(f"最终进入 OLS 的变量: {len(final_features)} 个\n")
    print("选中频率 top 15:")
    print(freq.head(15).round(3).to_string())

    # 构建 selected_mask（基于稳定变量，而非单次 LASSO）
    selected_mask = np.zeros(len(feature_names), dtype=bool)
    for name in final_features:
        selected_mask[feature_names.get_loc(name)] = True

    return {
        'freq': freq,
        'stable_features': stable_features,
        'final_features': final_features,
        'selected_mask': selected_mask
    }

In [27]:
def calculate_post_lasso_r2(X, y_aligned, selected_mask, selected_feature_names):
    X_selected = X[:, selected_mask]
    X_selected_const = sm.add_constant(X_selected)

    ols_model = sm.OLS(y_aligned, X_selected_const).fit(cov_type='HC3')
    r_squared = ols_model.rsquared
    r_squared_adj = ols_model.rsquared_adj
    ci = ols_model.conf_int()

    ols_results_df = pd.DataFrame({
        'O*NET_Activity': selected_feature_names,
        'OLS_Coefficient': ols_model.params[1:],
        'CI_Lower': ci[1:, 0],
        'CI_Upper': ci[1:, 1],
        'Std_Error': ols_model.bse[1:],
        'P_Value': ols_model.pvalues[1:],
        'Sig_10%': ols_model.pvalues[1:] < 0.10,
        'Sig_5%': ols_model.pvalues[1:] < 0.05
    }).sort_values('OLS_Coefficient', key=abs, ascending=False)

    return {
        'ols_model': ols_model,
        'r_squared': r_squared,
        'r_squared_adj': r_squared_adj,
        'ols_results_df': ols_results_df
    }

# Main

In [28]:
# # choose horizon here
# horizon_end = 12

# # Step 1. build CIR series
# cir_series = calculate_cir_series(
#     outcome='hourly_rate',
#     file_name='merged_occ_irf_trajectories.csv',
#     horizon_start=1,
#     horizon_end=horizon_end
# )

# print(cir_series)

# # Step 2. build y_series from mapping
# y_series = build_y_series_from_mapping(
#     cir_series=cir_series,
#     file_name='mapping_done.xlsx'
# )

# print(y_series.head())

# # Step 3. prepare O*NET data
# X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data(
#     y_series=y_series,
#     file_name='Work Activities.xlsx'
# )

# print(f"X shape: {X.shape}")
# print(f"Aligned samples: {len(aligned_idx)}")

# # Step 4. run LASSO
# lasso_results = run_lasso_selection(
#     X=X,
#     y_aligned=y_aligned,
#     feature_names=X_df.loc[aligned_idx].columns
# )

# print(f"Best alpha: {lasso_results['best_alpha']:.4f}")
# print(f"Number of non-zero coefficients: {np.sum(lasso_results['best_coefs'] != 0)}")
# print("Top 10 O*NET Activities:")
# for name, coef in zip(lasso_results['top_names'], lasso_results['top_coefs']):
#     sign = "+" if coef > 0 else "-"
#     print(f"  {sign} {name:<45} | coefficient: {coef:.4f}")

# stability_results = lasso_stability_check(
#     X=X,
#     y_aligned=y_aligned,
#     feature_names=X_df.loc[aligned_idx].columns
# )

# # Step 5. post-lasso OLS and R2
# post_lasso_results = calculate_post_lasso_r2(
#     X=X,
#     y_aligned=y_aligned,
#     selected_mask=stability_results['selected_mask'],
#     selected_feature_names=stability_results['final_features']
# )

# print("\nPost-Lasso OLS results:")
# print(f"R^2: {post_lasso_results['r_squared']:.4f} | Adjusted R^2: {post_lasso_results['r_squared_adj']:.4f}")
# print(f"Sample Size (n): {post_lasso_results['ols_model'].nobs}")

# print("\nPost-Lasso OLS coefficients:")
# print(post_lasso_results['ols_results_df'].to_string(
#     index=False,
#     formatters={
#         'OLS_Coefficient': '{:+.4f}'.format,
#         'Std_Error': '({:.4f})'.format,
#         'P_Value': '{:.4f}'.format
#     }
# ))


In [29]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='income',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
    y_series=y_series,
    file_names={
        'Activities': 'Work Activities.xlsx',
        'Skills':     'Skills.xlsx',
        'Abilities':  'Abilities.xlsx',
        'Knowledge':  'Knowledge.xlsx'
    },
    rti_path='../../data/occ1990dd_task_alm.dta',
    mapping_path='../../result/mapping/mapping_done.xlsx'
)

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0004 | Non-zero: 47
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 13 个
频率 >= 0.8 的变量 (高稳定): 24 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
Activities_Organizing, Planning, and Prioritizing Work    1.00
Abilities_Far Vision                                      0.99
Knowledge_Public Safety and Security                      0.97
RTI_index                                                 0.96
Skills_Installation                                       0.96
Knowledge_Mechanical                                      0.93
Knowledge_Geography                                       0.93
Knowledge_English Language                                0.93
Knowledge_Fine Arts                                       0.92
Knowledge_Law and Government                              0.92
Activities_Developing and Building Teams                  0.91
Activities_Getting Information                            0.91
Abilities_Rate 

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0007 | Non-zero: 46
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 9 个
频率 >= 0.8 的变量 (高稳定): 28 个
最终进入 OLS 的变量: 9 个

选中频率 top 15:
Abilities_Near Vision                                                1.00
Knowledge_Public Safety and Security                                 0.98
RTI_index                                                            0.98
Knowledge_Mechanical                                                 0.98
Activities_Organizing, Planning, and Prioritizing Work               0.97
Skills_Installation                                                  0.95
Abilities_Rate Control                                               0.95
Abilities_Far Vision                                                 0.93
Knowledge_Customer and Personal Service                              0.93
Knowledge_Economics and Accounting                                   0.88
Activities_Communicating with Supervisors, Peers, or Subordinates    0.87
Activities_Selling or 

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0022 | Non-zero: 48
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 10 个
频率 >= 0.8 的变量 (高稳定): 28 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
Knowledge_Public Safety and Security                      1.00
Activities_Organizing, Planning, and Prioritizing Work    0.98
Skills_Installation                                       0.94
RTI_index                                                 0.94
Knowledge_English Language                                0.93
Knowledge_Mechanical                                      0.92
Abilities_Far Vision                                      0.92
Abilities_Rate Control                                    0.91
Knowledge_Law and Government                              0.90
Activities_Selling or Influencing Others                  0.89
Knowledge_Economics and Accounting                        0.89
Knowledge_Mathematics                                     0.88
Activities_Gett

In [30]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='employment',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
    y_series=y_series,
    file_names={
        'Activities': 'Work Activities.xlsx',
        'Skills':     'Skills.xlsx',
        'Abilities':  'Abilities.xlsx',
        'Knowledge':  'Knowledge.xlsx'
    },
    rti_path='../../data/occ1990dd_task_alm.dta',
    mapping_path='../../result/mapping/mapping_done.xlsx'
)

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0002 | Non-zero: 76
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 21 个
频率 >= 0.8 的变量 (高稳定): 40 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Knowledge_Biology                                                1.00
Activities_Communicating with People Outside the Organization    1.00
Skills_Negotiation                                               1.00
Activities_Documenting/Recording Information                     0.99
Knowledge_Geography                                              0.99
Knowledge_Law and Government                                     0.99
Abilities_Memorization                                           0.99
Activities_Coordinating the Work and Activities of Others        0.99
Abilities_Manual Dexterity                                       0.98
Skills_Critical Thinking                                         0.97
Abilities_Dynamic Flexibility                                    0.96
Knowledge_Building and Construction                             

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0005 | Non-zero: 56
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 16 个
频率 >= 0.8 的变量 (高稳定): 33 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Knowledge_Biology                                                1.00
Activities_Communicating with People Outside the Organization    1.00
Skills_Negotiation                                               1.00
Skills_Critical Thinking                                         1.00
Activities_Documenting/Recording Information                     0.99
Abilities_Dynamic Flexibility                                    0.98
Knowledge_Law and Government                                     0.97
Skills_Management of Financial Resources                         0.97
Abilities_Depth Perception                                       0.96
Skills_Programming                                               0.95
Knowledge_Geography                                              0.95
Activities_Analyzing Data or Information                        

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0014 | Non-zero: 61
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 20 个
频率 >= 0.8 的变量 (高稳定): 43 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Knowledge_Law and Government                                     1.00
Knowledge_Biology                                                0.99
Skills_Negotiation                                               0.99
Skills_Critical Thinking                                         0.99
Activities_Documenting/Recording Information                     0.97
Activities_Communicating with People Outside the Organization    0.97
Abilities_Near Vision                                            0.97
Knowledge_Geography                                              0.96
Skills_Judgment and Decision Making                              0.96
Skills_Management of Financial Resources                         0.96
Abilities_Depth Perception                                       0.96
Knowledge_Economics and Accounting                              

In [31]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='hourly_rate',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
    y_series=y_series,
    file_names={
        'Activities': 'Work Activities.xlsx',
        'Skills':     'Skills.xlsx',
        'Abilities':  'Abilities.xlsx',
        'Knowledge':  'Knowledge.xlsx'
    },
    rti_path='../../data/occ1990dd_task_alm.dta',
    mapping_path='../../result/mapping/mapping_done.xlsx'
)

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0003 | Non-zero: 41
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 17 个
频率 >= 0.8 的变量 (高稳定): 30 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
Activities_Organizing, Planning, and Prioritizing Work    0.99
Abilities_Far Vision                                      0.98
Knowledge_Mechanical                                      0.97
Abilities_Rate Control                                    0.96
Abilities_Explosive Strength                              0.96
RTI_index                                                 0.96
Knowledge_English Language                                0.95
Knowledge_Customer and Personal Service                   0.95
Knowledge_Public Safety and Security                      0.93
Knowledge_Food Production                                 0.93
Skills_Installation                                       0.92
Skills_Repairing                                          0.92
Activities_Coac

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0007 | Non-zero: 40
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 13 个
频率 >= 0.8 的变量 (高稳定): 27 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
Knowledge_Mechanical                                      1.00
RTI_index                                                 0.98
Activities_Organizing, Planning, and Prioritizing Work    0.97
Knowledge_Customer and Personal Service                   0.97
Abilities_Far Vision                                      0.96
Abilities_Rate Control                                    0.94
Abilities_Dynamic Flexibility                             0.94
Skills_Repairing                                          0.94
Skills_Installation                                       0.94
Knowledge_Public Safety and Security                      0.93
Abilities_Explosive Strength                              0.92
Knowledge_Food Production                                 0.91
Activities_Coac

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0022 | Non-zero: 42
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 12 个
频率 >= 0.8 的变量 (高稳定): 24 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
Knowledge_Mechanical                                      1.00
Activities_Organizing, Planning, and Prioritizing Work    0.99
Abilities_Far Vision                                      0.98
Knowledge_Public Safety and Security                      0.96
Knowledge_Customer and Personal Service                   0.94
Knowledge_English Language                                0.93
RTI_index                                                 0.93
Knowledge_Mathematics                                     0.93
Abilities_Rate Control                                    0.92
Skills_Installation                                       0.91
Skills_Repairing                                          0.91
Abilities_Explosive Strength                              0.89
Activities_Coac

In [32]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='hours',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
    y_series=y_series,
    file_names={
        'Activities': 'Work Activities.xlsx',
        'Skills':     'Skills.xlsx',
        'Abilities':  'Abilities.xlsx',
        'Knowledge':  'Knowledge.xlsx'
    },
    rti_path='../../data/occ1990dd_task_alm.dta',
    mapping_path='../../result/mapping/mapping_done.xlsx'
)

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0001 | Non-zero: 38
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 15 个
频率 >= 0.8 的变量 (高稳定): 29 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Skills_Persuasion                                                  1.00
Knowledge_Biology                                                  0.99
Abilities_Dynamic Flexibility                                      0.98
Knowledge_Public Safety and Security                               0.98
Knowledge_Administrative                                           0.97
Knowledge_Food Production                                          0.97
Activities_Developing and Building Teams                           0.96
Activities_Assisting and Caring for Others                         0.96
Abilities_Gross Body Equilibrium                                   0.95
Activities_Operating Vehicles, Mechanized Devices, or Equipment    0.94
Knowledge_Geography                                                0.92
Skills_Equipment Selection                

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0001 | Non-zero: 65
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 17 个
频率 >= 0.8 的变量 (高稳定): 38 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Knowledge_Public Safety and Security                               1.00
Skills_Persuasion                                                  1.00
Skills_Management of Financial Resources                           0.98
Knowledge_Biology                                                  0.97
Activities_Operating Vehicles, Mechanized Devices, or Equipment    0.97
Abilities_Gross Body Equilibrium                                   0.95
RTI_index                                                          0.94
Abilities_Category Flexibility                                     0.93
Abilities_Dynamic Flexibility                                      0.92
Activities_Training and Teaching Others                            0.92
Activities_Selling or Influencing Others                           0.92
Skills_Equipment Selection                

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0005 | Non-zero: 61
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 16 个
频率 >= 0.8 的变量 (高稳定): 43 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Skills_Persuasion                                                  1.00
Knowledge_Public Safety and Security                               0.98
Knowledge_Biology                                                  0.96
Skills_Operations Analysis                                         0.95
Activities_Operating Vehicles, Mechanized Devices, or Equipment    0.94
Knowledge_Fine Arts                                                0.94
Abilities_Gross Body Equilibrium                                   0.94
Activities_Getting Information                                     0.93
Activities_Training and Teaching Others                            0.93
Skills_Technology Design                                           0.93
Skills_Management of Financial Resources                           0.93
Abilities_Deductive Reasoning             

In [33]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='income_share',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
    y_series=y_series,
    file_names={
        'Activities': 'Work Activities.xlsx',
        'Skills':     'Skills.xlsx',
        'Abilities':  'Abilities.xlsx',
        'Knowledge':  'Knowledge.xlsx'
    },
    rti_path='../../data/occ1990dd_task_alm.dta',
    mapping_path='../../result/mapping/mapping_done.xlsx'
)

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0001 | Non-zero: 50
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 11 个
频率 >= 0.8 的变量 (高稳定): 30 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Skills_Science                                               1.00
Skills_Management of Financial Resources                     1.00
Knowledge_Fine Arts                                          0.97
Activities_Performing Administrative Activities              0.95
Activities_Repairing and Maintaining Electronic Equipment    0.94
Skills_Persuasion                                            0.94
Activities_Monitoring and Controlling Resources              0.93
Activities_Getting Information                               0.92
Knowledge_History and Archeology                             0.91
Activities_Providing Consultation and Advice to Others       0.91
Knowledge_Customer and Personal Service                      0.91
Skills_Complex Problem Solving                               0.88
Abilities_Auditory Attention              

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0002 | Non-zero: 45
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 10 个
频率 >= 0.8 的变量 (高稳定): 31 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Skills_Science                                               1.00
Skills_Management of Financial Resources                     1.00
Activities_Performing Administrative Activities              0.95
Skills_Persuasion                                            0.94
Knowledge_Fine Arts                                          0.93
Knowledge_History and Archeology                             0.93
Activities_Monitoring and Controlling Resources              0.92
Activities_Repairing and Maintaining Electronic Equipment    0.91
Activities_Getting Information                               0.90
Knowledge_Customer and Personal Service                      0.90
Skills_Complex Problem Solving                               0.87
Activities_Providing Consultation and Advice to Others       0.87
Skills_Negotiation                        

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0006 | Non-zero: 41
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 8 个
频率 >= 0.8 的变量 (高稳定): 25 个
最终进入 OLS 的变量: 8 个

选中频率 top 15:
Skills_Science                                               1.00
Skills_Management of Financial Resources                     1.00
Knowledge_Fine Arts                                          0.94
Activities_Monitoring and Controlling Resources              0.94
Skills_Persuasion                                            0.94
Activities_Performing Administrative Activities              0.93
Knowledge_History and Archeology                             0.91
Activities_Repairing and Maintaining Electronic Equipment    0.90
Knowledge_Building and Construction                          0.89
Skills_Complex Problem Solving                               0.89
Knowledge_English Language                                   0.87
Knowledge_Customer and Personal Service                      0.87
Skills_Operation and Control                

In [34]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='inequality',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
    y_series=y_series,
    file_names={
        'Activities': 'Work Activities.xlsx',
        'Skills':     'Skills.xlsx',
        'Abilities':  'Abilities.xlsx',
        'Knowledge':  'Knowledge.xlsx'
    },
    rti_path='../../data/occ1990dd_task_alm.dta',
    mapping_path='../../result/mapping/mapping_done.xlsx'
)

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0006 | Non-zero: 30
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 13 个
频率 >= 0.8 的变量 (高稳定): 28 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                        1.00
Knowledge_Mechanical                                         0.99
Activities_Coordinating the Work and Activities of Others    0.98
Abilities_Fluency of Ideas                                   0.96
Activities_Selling or Influencing Others                     0.95
Knowledge_Public Safety and Security                         0.94
Knowledge_Production and Processing                          0.94
Abilities_Far Vision                                         0.93
Abilities_Spatial Orientation                                0.93
Abilities_Category Flexibility                               0.93
Skills_Programming                                           0.92
Skills_Negotiation                                           0.92
Abilities_Dynamic Flexibility             

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0013 | Non-zero: 40
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 7 个
频率 >= 0.8 的变量 (高稳定): 17 个
最终进入 OLS 的变量: 7 个

选中频率 top 15:
Abilities_Near Vision                                        1.00
Knowledge_Public Safety and Security                         0.99
Knowledge_Mechanical                                         0.99
Abilities_Far Vision                                         0.98
Activities_Coordinating the Work and Activities of Others    0.94
Activities_Organizing, Planning, and Prioritizing Work       0.91
Activities_Selling or Influencing Others                     0.90
Skills_Installation                                          0.89
RTI_index                                                    0.88
Activities_Coaching and Developing Others                    0.86
Knowledge_Mathematics                                        0.86
Knowledge_Production and Processing                          0.85
Skills_Monitoring                           

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0035 | Non-zero: 37
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 7 个
频率 >= 0.8 的变量 (高稳定): 20 个
最终进入 OLS 的变量: 7 个

选中频率 top 15:
Abilities_Near Vision                                        1.00
Knowledge_Public Safety and Security                         1.00
Knowledge_Mechanical                                         0.99
Abilities_Far Vision                                         0.96
Knowledge_Law and Government                                 0.94
Activities_Selling or Influencing Others                     0.93
Activities_Coordinating the Work and Activities of Others    0.92
Activities_Getting Information                               0.89
Knowledge_Production and Processing                          0.88
Activities_Organizing, Planning, and Prioritizing Work       0.87
RTI_index                                                    0.87
Skills_Installation                                          0.87
Skills_Monitoring                           

In [35]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='median',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
    y_series=y_series,
    file_names={
        'Activities': 'Work Activities.xlsx',
        'Skills':     'Skills.xlsx',
        'Abilities':  'Abilities.xlsx',
        'Knowledge':  'Knowledge.xlsx'
    },
    rti_path='../../data/occ1990dd_task_alm.dta',
    mapping_path='../../result/mapping/mapping_done.xlsx'
)

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0003 | Non-zero: 63
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 12 个
频率 >= 0.8 的变量 (高稳定): 32 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
Activities_Organizing, Planning, and Prioritizing Work    1.00
Knowledge_Mechanical                                      1.00
RTI_index                                                 0.99
Abilities_Far Vision                                      0.99
Knowledge_Geography                                       0.98
Skills_Installation                                       0.95
Abilities_Stamina                                         0.93
Knowledge_English Language                                0.93
Abilities_Finger Dexterity                                0.92
Skills_Repairing                                          0.90
Knowledge_Public Safety and Security                      0.90
Knowledge_Communications and Media                        0.89
Abilities_Speed

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0005 | Non-zero: 59
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 17 个
频率 >= 0.8 的变量 (高稳定): 28 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
RTI_index                                                 1.00
Knowledge_Mechanical                                      1.00
Activities_Organizing, Planning, and Prioritizing Work    0.97
Abilities_Finger Dexterity                                0.97
Abilities_Far Vision                                      0.96
Skills_Installation                                       0.96
Knowledge_Public Safety and Security                      0.96
Abilities_Stamina                                         0.95
Knowledge_Economics and Accounting                        0.95
Knowledge_Food Production                                 0.94
Abilities_Rate Control                                    0.93
Knowledge_Customer and Personal Service                   0.91
Knowledge_Commu

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0014 | Non-zero: 71
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 16 个
频率 >= 0.8 的变量 (高稳定): 32 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                        1.00
Knowledge_Public Safety and Security                         0.99
Knowledge_Mechanical                                         0.97
RTI_index                                                    0.97
Abilities_Far Vision                                         0.96
Abilities_Rate Control                                       0.95
Abilities_Speed of Limb Movement                             0.95
Knowledge_Food Production                                    0.95
Skills_Installation                                          0.95
Abilities_Explosive Strength                                 0.94
Activities_Repairing and Maintaining Mechanical Equipment    0.94
Knowledge_Economics and Accounting                           0.93
Knowledge_Law and Government              

In [ ]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='unemployment',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
    y_series=y_series,
    file_names={
        'Activities': 'Work Activities.xlsx',
        'Skills':     'Skills.xlsx',
        'Abilities':  'Abilities.xlsx',
        'Knowledge':  'Knowledge.xlsx'
    },
    rti_path='../../data/occ1990dd_task_alm.dta',
    mapping_path='../../result/mapping/mapping_done.xlsx'
)

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0001 | Non-zero: 70
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 24 个
频率 >= 0.8 的变量 (高稳定): 38 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Dynamic Flexibility                                                              1.00
Activities_Drafting, Laying Out, and Specifying Technical Devices, Parts, and Equipment    1.00
Knowledge_Building and Construction                                                        1.00
Activities_Developing and Building Teams                                                   1.00
Activities_Coordinating the Work and Activities of Others                                  1.00
Activities_Working with Computers                                                          0.99
Skills_Equipment Selection                                                                 0.99
Abilities_Far Vision                                                                       0.99
Knowledge_Administrative                                          

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0001 | Non-zero: 67
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 20 个
频率 >= 0.8 的变量 (高稳定): 34 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Dynamic Flexibility                                                              1.00
Knowledge_Administrative                                                                   1.00
Skills_Equipment Selection                                                                 0.99
Activities_Communicating with People Outside the Organization                              0.98
Activities_Documenting/Recording Information                                               0.98
Knowledge_Biology                                                                          0.97
Abilities_Multilimb Coordination                                                           0.97
Activities_Working with Computers                                                          0.96
Knowledge_Building and Construction                               

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0003 | Non-zero: 70
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 22 个
频率 >= 0.8 的变量 (高稳定): 37 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Dynamic Flexibility                                                              1.00
Knowledge_Administrative                                                                   1.00
Skills_Equipment Selection                                                                 0.98
Knowledge_Biology                                                                          0.97
Activities_Documenting/Recording Information                                               0.97
Knowledge_Computers and Electronics                                                        0.97
Knowledge_Chemistry                                                                        0.97
Activities_Communicating with People Outside the Organization                              0.96
Knowledge_Food Production                                         

In [38]:
outcomes = [
    'unemployment', 'employment', 'income', 'hourly_rate',
    'hours', 'income_share', 'inequality', 'median'
]

all_outcomes_results = {}

for outcome in outcomes:
    print(f"\n{'#'*60}")
    print(f"OUTCOME: {outcome}")
    print(f"{'#'*60}")

    all_horizon_results = {}

    for horizon_end in [6, 12, 36]:
        print(f"\n{'='*50}")
        print(f"Horizon {horizon_end}")
        print(f"{'='*50}")

        cir_series = calculate_cir_series(
            outcome=outcome,
            file_name='merged_occ_irf_trajectories.csv',
            horizon_end=horizon_end
        )

        y_series = build_y_series_from_mapping(
            cir_series=cir_series,
            file_name='mapping_done.xlsx'
        )

        X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data_extended(
            y_series=y_series,
            file_names={
                'Activities': 'Work Activities.xlsx',
                'Skills':     'Skills.xlsx',
                'Abilities':  'Abilities.xlsx',
                'Knowledge':  'Knowledge.xlsx'
            },
            rti_path='../../data/occ1990dd_task_alm.dta',
            mapping_path='../../result/mapping/mapping_done.xlsx'
        )

        lasso_results = run_lasso_selection(
            X=X,
            y_aligned=y_aligned,
            feature_names=X_df.loc[aligned_idx].columns
        )
        print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")

        stability_results = lasso_stability_check(
            X=X,
            y_aligned=y_aligned,
            feature_names=X_df.loc[aligned_idx].columns,
            freq_threshold=0.9
        )

        post_lasso_results = calculate_post_lasso_r2(
            X=X,
            y_aligned=y_aligned,
            selected_mask=stability_results['selected_mask'],
            selected_feature_names=stability_results['final_features']
        )

        print(f"R^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
        print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
        print("\nOLS coefficients:")
        print(post_lasso_results['ols_results_df'].to_string(
            index=False,
            formatters={
                'OLS_Coefficient': '{:+.4f}'.format,
                'Std_Error': '({:.4f})'.format,
                'P_Value': '{:.4f}'.format
            }
        ))

        all_horizon_results[horizon_end] = {
            'freq':           stability_results['freq'],
            'final_features': stability_results['final_features'],
            'ols_df':         post_lasso_results['ols_results_df'],
            'r2':             post_lasso_results['r_squared'],
            'adj_r2':         post_lasso_results['r_squared_adj']
        }

    all_outcomes_results[outcome] = all_horizon_results

# ── 汇总表 ────────────────────────────────────────────────────
# 表1：R² 汇总（outcome × horizon）
r2_rows = []
for outcome in outcomes:
    row = {'outcome': outcome}
    for h in [6, 12, 36]:
        row[f'R2_h{h}']    = round(all_outcomes_results[outcome][h]['r2'], 4)
        row[f'AdjR2_h{h}'] = round(all_outcomes_results[outcome][h]['adj_r2'], 4)
    r2_rows.append(row)

r2_summary = pd.DataFrame(r2_rows).set_index('outcome')
print(f"\n{'='*60}")
print("汇总表 1：R² 和 Adjusted R²")
print(f"{'='*60}")
print(r2_summary.to_string())

# 表2：跨 outcome × horizon 的变量稳定性
print(f"\n{'='*60}")
print("汇总表 2：各 outcome 下三个 horizon 均选中的稳定变量")
print(f"{'='*60}")

for outcome in outcomes:
    h_results = all_outcomes_results[outcome]
    # 三个 horizon 都选中的变量
    stable = set(h_results[6]['final_features']) \
           & set(h_results[12]['final_features']) \
           & set(h_results[36]['final_features'])
    print(f"\n{outcome} ({len(stable)} 个稳定变量):")
    for v in sorted(stable):
        freqs = [round(h_results[h]['freq'].get(v, 0), 2) for h in [6, 12, 36]]
        print(f"  {v:<55} freq: {freqs}")

# 表3：完整系数表（每个 outcome × horizon 的 OLS 结果）
print(f"\n{'='*60}")
print("汇总表 3：完整 OLS 系数（所有 outcome × horizon）")
print(f"{'='*60}")

coef_rows = []
for outcome in outcomes:
    for h in [6, 12, 36]:
        ols_df = all_outcomes_results[outcome][h]['ols_df']
        for _, row in ols_df.iterrows():
            coef_rows.append({
                'outcome':    outcome,
                'horizon':    h,
                'feature':    row['O*NET_Activity'],
                'coef':       round(row['OLS_Coefficient'], 4),
                'pvalue':     round(row['P_Value'], 4),
                'sig_5pct':   row['Sig_5%'],
                'r2':         round(all_outcomes_results[outcome][h]['r2'], 4)
            })

coef_summary = pd.DataFrame(coef_rows)
print(coef_summary.to_string(index=False))

# CSV 导出（方便后续处理）
r2_summary.to_csv('summary_r2.csv')
coef_summary.to_csv('summary_coefficients.csv', index=False)
print("\n已导出 summary_r2.csv 和 summary_coefficients.csv")


############################################################
OUTCOME: unemployment
############################################################

Horizon 6
RTI 文件中有效 occ1990dd: 330 个
mapping 过滤后剩余行数: 776
有 RTI 的 SOC-2018 数量: 687
O*NET 合并后: 614 个 SOC, 161 个特征
拼入 RTI 后: 614 个 SOC, 162 个特征（含 RTI）
X shape: (609, 162) | Aligned samples: 609


/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0001 | Non-zero: 70
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 24 个
频率 >= 0.8 的变量 (高稳定): 38 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Dynamic Flexibility                                                              1.00
Activities_Drafting, Laying Out, and Specifying Technical Devices, Parts, and Equipment    1.00
Knowledge_Building and Construction                                                        1.00
Activities_Developing and Building Teams                                                   1.00
Activities_Coordinating the Work and Activities of Others                                  1.00
Activities_Working with Computers                                                          0.99
Skills_Equipment Selection                                                                 0.99
Abilities_Far Vision                                                                       0.99
Knowledge_Administrative                                          

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0001 | Non-zero: 67
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 20 个
频率 >= 0.8 的变量 (高稳定): 34 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Dynamic Flexibility                                                              1.00
Knowledge_Administrative                                                                   1.00
Skills_Equipment Selection                                                                 0.99
Activities_Communicating with People Outside the Organization                              0.98
Activities_Documenting/Recording Information                                               0.98
Knowledge_Biology                                                                          0.97
Abilities_Multilimb Coordination                                                           0.97
Activities_Working with Computers                                                          0.96
Knowledge_Building and Construction                               

...

Best alpha: 0.0003 | Non-zero: 63
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 12 个
频率 >= 0.8 的变量 (高稳定): 32 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
Activities_Organizing, Planning, and Prioritizing Work    1.00
Knowledge_Mechanical                                      1.00
RTI_index                                                 0.99
Abilities_Far Vision                                      0.99
Knowledge_Geography                                       0.98
Skills_Installation                                       0.95
Abilities_Stamina                                         0.93
Knowledge_English Language                                0.93
Abilities_Finger Dexterity                                0.92
Skills_Repairing                                          0.90
Knowledge_Public Safety and Security                      0.90
Knowledge_Communications and Media                        0.89
Abilities_Speed

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0005 | Non-zero: 59
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 17 个
频率 >= 0.8 的变量 (高稳定): 28 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                     1.00
RTI_index                                                 1.00
Knowledge_Mechanical                                      1.00
Activities_Organizing, Planning, and Prioritizing Work    0.97
Abilities_Finger Dexterity                                0.97
Abilities_Far Vision                                      0.96
Skills_Installation                                       0.96
Knowledge_Public Safety and Security                      0.96
Abilities_Stamina                                         0.95
Knowledge_Economics and Accounting                        0.95
Knowledge_Food Production                                 0.94
Abilities_Rate Control                                    0.93
Knowledge_Customer and Personal Service                   0.91
Knowledge_Commu

/home/banjiayu/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


Best alpha: 0.0014 | Non-zero: 71
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 16 个
频率 >= 0.8 的变量 (高稳定): 32 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Abilities_Near Vision                                        1.00
Knowledge_Public Safety and Security                         0.99
Knowledge_Mechanical                                         0.97
RTI_index                                                    0.97
Abilities_Far Vision                                         0.96
Abilities_Rate Control                                       0.95
Abilities_Speed of Limb Movement                             0.95
Knowledge_Food Production                                    0.95
Skills_Installation                                          0.95
Abilities_Explosive Strength                                 0.94
Activities_Repairing and Maintaining Mechanical Equipment    0.94
Knowledge_Economics and Accounting                           0.93
Knowledge_Law and Government              